# Qwen3.6-35B-A3B → 64-Layer Looped MoE Transformer (`TassP0lar/Qwen3.6-35B-A3B-Loop64`)

**Single Self-Resuming End-to-End Colab Notebook (G4 · RTX PRO 6000 Blackwell 96 GB · 230 GB SSD)**

- **64-Virtual-Layer Encode–Think–Decode (`8 + 24×2 + 8`)**: Keeps prelude `[0,8)` & coda `[32,40)` single-pass, loops the 24-layer core `[8,32)` with an **mHC Birkhoff bridge** (zero-gated step-0 bit-exactness), isolated `(layer, pass)` **Gated DeltaNet** state slots, **MoR recursive KV + D-LSA bucket sharing** (1M KV stays 10.24 GB FP8), and **1M YaRN + Distilled Latent Sparse Attention (D-LSA)**.
- **Cross-Pass Dynamic Expert Re-Routing (Beyond 3B Active Params)**: Learnable per-pass router adapters (`PassRouterAdapters`) and cross-pass expert repulsion allow Pass 2+ to wake up distinct Top-8 experts from the frozen 32.2B MoE pool on every recurrent pass.
- **Zero-Drive HF Commit Sync (`TassP0lar/Qwen3.6-35B-A3B-Loop64`)**: Add `HF_TOKEN` (Write permission) in Colab `Secrets`. Every checkpoint, ESFT scan, and benchmark scorecard commits atomically to Hugging Face Hub (`manifest.json` + `README.md`), then prunes superseded local files so disk stays well under 230 GB. **Pressing `Runtime → Run all` in any new Colab session automatically resumes from the latest HF commit.**
- **Verbose Crash Diagnostics**: Every command runs in an isolated subprocess and writes `crash/crash_<ts>.md` (`Hints`, `Traceback`, per-frame tensor shapes/NaNs, CUDA memory). Set `DEBUG_LEVEL = 2` (NaN hooks + C++ traces) or `3` (`CUDA_LAUNCH_BLOCKING=1` + autograd anomaly + memory timeline).

In [ ]:
REPO = "https://github.com/BaiTian6641/Qwen3.6-35B-A3B-Loop60.git"
BRANCH = "main"
!test -d /content/qloop || git clone -b $BRANCH $REPO /content/qloop
%cd /content/qloop
!git pull --ff-only
!bash scripts/colab_setup.sh

In [ ]:
from qloop.colab import hub_status, q, reclaim_disk, run_pipeline, setup_env, tail_metrics

HF_REPO = "TassP0lar/Qwen3.6-35B-A3B-Loop64"  # public HF repo synced via Colab Secret HF_TOKEN
DEBUG_LEVEL = 1                               # 1=normal | 2=verbose NaN hooks | 3=forensic CUDA_LAUNCH_BLOCKING
TIME_BUDGET_HOURS = 21.5                      # clean checkpoint + HF commit before Colab 24h timeout

setup_env(hf_repo=HF_REPO, run_root="/content/runs", hf_home="/content/hf_cache")
q("doctor --freeze requirements/colab.lock.txt --model Qwen/Qwen3.6-35B-A3B")
hub_status(HF_REPO)

### 1 · End-to-End Self-Resuming Pipeline (`s0_probe → s1_loop_heal → s1_rescan → s2_sft → s3_long_lsa → s4_rl → eval`)
Queries `manifest.json` on `TassP0lar/Qwen3.6-35B-A3B-Loop64`, skips completed stages, pulls the latest trainable checkpoint (`~1–3 GB`) on fresh sessions, commits progress by stage, and reclaims local SSD space automatically.

In [ ]:
run_pipeline(
    hf_repo=HF_REPO,
    debug=DEBUG_LEVEL,
    time_budget_hours=TIME_BUDGET_HOURS,
)
hub_status(HF_REPO)

### 2 · Optional: Single-Stage Forensic Debug & Live Telemetry (`RUN_MANUAL_STAGE = True`)
Use this cell when tuning core ranges, inspecting `eval/depth_gain` & `loop/reroute_rate`, or re-running one stage with `--debug 2` / `--debug 3`.

In [ ]:
RUN_MANUAL_STAGE = False
MANUAL_STAGE = "s1_loop_heal"  # s1_loop_heal | s2_sft | s3_long_lsa | s4_rl
MANUAL_DEBUG = 2               # 2=NaN/Inf module hooks | 3=CUDA_LAUNCH_BLOCKING + anomaly

if RUN_MANUAL_STAGE:
    q(f"train -c configs/stages/{MANUAL_STAGE}.yaml --debug {MANUAL_DEBUG} -s checkpoint.sync.hf_repo={HF_REPO}")
    tail_metrics(MANUAL_STAGE)
    reclaim_disk(keep=MANUAL_STAGE)

### 3 · Optional: Test-Time Depth Extrapolation (`40L` vs `64L` vs `88L`) & Container Benchmarks (`RUN_EXTRA_EVAL = True`)

In [ ]:
RUN_EXTRA_EVAL = False

if RUN_EXTRA_EVAL:
    # Sweep 1 pass (40L), 2 passes (64L), and 3 passes (88L zero-shot depth extrapolation) on ARC-AGI-2 & AIME 2026
    q(
        "eval -c configs/eval/suite.yaml --ckpt /content/runs/s4_rl "
        "-s loop.num_passes=3 -s loop.target_virtual_layers=88 "
        "-s eval.defaults.depths=[1,2,3] "
        f"-s checkpoint.sync.hf_repo={HF_REPO}"
    )

### 4 · Optional: 1-Call Inference & Dynamic Expert Re-Routing Check from Hugging Face Hub (`RUN_HUB_INFERENCE = True`)

In [ ]:
RUN_HUB_INFERENCE = False

if RUN_HUB_INFERENCE:
    from qloop.eval.generate import Generator
    from qloop.hub import load_qloop_from_hub

    stack = load_qloop_from_hub(HF_REPO)
    gen = Generator(stack.model, stack.tokenizer, loop_controller=stack.loop)
    prompt = "Let a, b, c > 0 with a + b + c = 3. Prove that a^2/(a+2b^2) + b^2/(b+2c^2) + c^2/(c+2a^2) >= 1."
    for depth in (1, 2):
        with gen.depth(depth):
            out = gen.generate([prompt], max_new_tokens=1024, temperature=0.0)[0]
        v_layers = 40 if depth == 1 else 64
        print(f"=== Depth {depth} ({v_layers} virtual layers) ===\n{out.text[:800]}\n")
        if depth == 2 and stack.loop is not None:
            print("Loop & Expert Routing Stats:", stack.loop.stats())